In [1]:
import time
import heapq
import numpy as np
from scipy.sparse import load_npz
from scipy.sparse.csgraph import dijkstra as scipy_dijkstra
from scipy.spatial import cKDTree

# 1. Load đồ thị từ ổ cứng lên RAM
t0 = time.perf_counter()
adj_matrix = load_npz("data/hcm_center_moto_adj.npz")
meta = np.load("data/hcm_center_moto_meta.npz")
coords = meta["coords"]      # Shape (14104, 2) -> [lat, lng]
node_ids = meta["node_ids"]
t_load = (time.perf_counter() - t0) * 1000
print(f"Thời gian load đồ thị ({adj_matrix.shape[0]:,} nút) lên RAM: {t_load:.2f} ms")

# 2. Dựng cây tìm kiếm không gian KDTree để snap tọa độ GPS người dùng về đỉnh gần nhất
tree = cKDTree(coords)

# Giả sử vị trí GPS thật của 4 người bạn ở Q1, Q3, Bình Thạnh
people_gps = {
    "An (Chợ Bến Thành - Q1)": (10.7725, 106.6980),
    "Bình (Hồ Con Rùa - Q3)": (10.7827, 106.6959),
    "Chi (Hàng Xanh - Bình Thạnh)": (10.8013, 106.7114),
    "Phi (Vạn Kiếp - Bình Thạnh)": (10.7995, 106.6922),
}

# Tìm đỉnh gần nhất cho từng người
source_indices = []
for name, gps in people_gps.items():
    dist_deg, node_idx = tree.query(gps)
    source_indices.append(node_idx)
    print(f"- {name} -> Snap vào đỉnh #{node_idx} (tọa độ: {coords[node_idx][0]:.5f}, {coords[node_idx][1]:.5f})")

# 3. BENCHMARK 1: Chạy SciPy Dijkstra (Cython/C) cho cả 4 người cùng lúc
t0 = time.perf_counter()
# directed=True vì đồ thị đường phố có đường 1 chiều!
T_scipy = scipy_dijkstra(csgraph=adj_matrix, directed=True, indices=source_indices)
t_scipy = (time.perf_counter() - t0) * 1000
print(f"\n[SciPy C-engine] Tính trường thời gian cho 4 người trên 14,104 đỉnh mất: {t_scipy:.2f} ms")

# 4. BENCHMARK 2: Chuyển sang dict và chạy Dijkstra Python thuần (heapq) từ tuần trước
graph_dict = {i: {} for i in range(adj_matrix.shape[0])}
rows, cols = adj_matrix.nonzero()
for u, v, w in zip(rows, cols, adj_matrix.data):
    graph_dict[int(u)][int(v)] = float(w)

def dijkstra_python(graph, start):
    dist = np.full(len(graph), np.inf, dtype=np.float64)
    dist[start] = 0.0
    pq = [(0.0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in graph[u].items():
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

t0 = time.perf_counter()
T_python = np.array([dijkstra_python(graph_dict, idx) for idx in source_indices])
t_python = (time.perf_counter() - t0) * 1000
print(f"[Python heapq]   Tính trường thời gian cho 4 người trên 14,104 đỉnh mất: {t_python:.2f} ms")

# 5. Kiểm chứng sai số giữa 2 cách cài đặt
max_diff = np.max(np.abs(T_scipy - T_python))
print(f"\nTốc độ SciPy nhanh hơn Python thuần: {t_python / t_scipy:.1f} lần")
print(f"Sai số lớn nhất giữa 2 kết quả: {max_diff:.6f} giây")

Thời gian load đồ thị (14,104 nút) lên RAM: 30.74 ms
- An (Chợ Bến Thành - Q1) -> Snap vào đỉnh #1708 (tọa độ: 10.77219, 106.69756)
- Bình (Hồ Con Rùa - Q3) -> Snap vào đỉnh #11523 (tọa độ: 10.78301, 106.69592)
- Chi (Hàng Xanh - Bình Thạnh) -> Snap vào đỉnh #2081 (tọa độ: 10.80119, 106.71138)
- Phi (Vạn Kiếp - Bình Thạnh) -> Snap vào đỉnh #8430 (tọa độ: 10.79957, 106.69217)

[SciPy C-engine] Tính trường thời gian cho 4 người trên 14,104 đỉnh mất: 7.53 ms
[Python heapq]   Tính trường thời gian cho 4 người trên 14,104 đỉnh mất: 88.18 ms

Tốc độ SciPy nhanh hơn Python thuần: 11.7 lần
Sai số lớn nhất giữa 2 kết quả: 0.000000 giây
